In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

%matplotlib widget
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 120,
})

REPO_ROOT = Path(".").resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "pyproject.toml").exists() or (REPO_ROOT / ".git").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
MEAS = REPO_ROOT / "measurements"

CAMPAIGN = "MBB/2026-03-10_max_speed_idle"
SESSIONS = {
    "A1 (200 GeV)": {
        "results_body": f"{CAMPAIGN}/20260310_170427_SPS_MBB/20260310_170449_MBB/20260310_170449_MBB_Run_00_I_0.00A_body_results.txt",
        "results_fringe": f"{CAMPAIGN}/20260310_170427_SPS_MBB/20260310_170449_MBB/20260310_170449_MBB_Run_00_I_0.00A_fringe_results.txt",
    },
    "B1 (26 GeV)": {
        "results_body": f"{CAMPAIGN}/20260310_172501_SPS_MBB/20260310_172522_MBB/20260310_172522_MBB_Run_00_I_0.00A_body_results.txt",
        "results_fringe": f"{CAMPAIGN}/20260310_172501_SPS_MBB/20260310_172522_MBB/20260310_172522_MBB_Run_00_I_0.00A_fringe_results.txt",
    },
    "B2 (26 GeV)": {
        "results_body": f"{CAMPAIGN}/20260310_174733_SPS_MBB/20260310_174754_MBB/20260310_174754_MBB_Run_00_I_0.00A_body_results.txt",
        "results_fringe": f"{CAMPAIGN}/20260310_174733_SPS_MBB/20260310_174754_MBB/20260310_174754_MBB_Run_00_I_0.00A_fringe_results.txt",
    },
    "A2 (200 GeV)": {
        "results_body": f"{CAMPAIGN}/20260310_180839_SPS_MBB/20260310_180902_MBB/20260310_180902_MBB_Run_00_I_0.00A_body_results.txt",
        "results_fringe": f"{CAMPAIGN}/20260310_180839_SPS_MBB/20260310_180902_MBB/20260310_180902_MBB_Run_00_I_0.00A_fringe_results.txt",
    },
}
SESSION_NAMES = list(SESSIONS.keys())

COLORS = {
    "A1 (200 GeV)": "tab:blue",
    "B1 (26 GeV)":  "tab:orange",
    "B2 (26 GeV)":  "tab:green",
    "A2 (200 GeV)": "tab:red",
}


def load_turns(path_key, cfg):
    fpath = MEAS / cfg[path_key]
    assert fpath.exists(), f"Missing: {fpath}"
    df = pd.read_csv(fpath, sep="\t")
    df = df.rename(columns={
        "Time(s)": "t_s", "Duration(s)": "dur_s", "I(A)": "I_A",
        "Ramprate(A/s)": "rr",
        "B_main(T)": "B1_T", "A_main(T)": "A1_T",
        "b2(Units)": "b2", "a2(Units)": "a2",
        "b3(Units)": "b3", "a3(Units)": "a3",
    })
    df["turn"] = np.arange(len(df))
    df["B1_T"] = -df["B1_T"]
    if "A1_T" in df.columns:
        df["A1_T"] = -df["A1_T"]
    for n in range(2, 16):
        if n % 2 == 0:
            for prefix in ["b", "a"]:
                for col in [f"{prefix}{n}", f"{prefix}{n}(Units)"]:
                    if col in df.columns:
                        df[col] = -df[col]
    return df


def find_phases(df, std_threshold=4000, idle_threshold=200):
    """Split a session into 'std', 'md1', 'meas' slices.

    Measurement starts at the idle after the last MD1 cycle
    (i.e. the idle between MD1 and SFTPRO).
    """
    I = df["I_A"].values
    above = I > std_threshold
    peaks, in_peak = [], False
    for j in range(len(I)):
        if above[j] and not in_peak:
            in_peak, pidx, pmax = True, j, I[j]
        elif above[j] and in_peak:
            if I[j] > pmax:
                pidx, pmax = j, I[j]
        elif not above[j] and in_peak:
            peaks.append(pidx)
            in_peak = False
    if len(peaks) < 11:
        return {"std": slice(0, 0), "md1": slice(0, len(I)), "meas": slice(len(I), len(I))}
    last_std = peaks[9]
    after = np.where(I[last_std:] < idle_threshold)[0]
    md1_start = last_std + after[0] if len(after) > 0 else last_std + 10
    meas_peak = peaks[10]
    # Find last idle point before SFTPRO ramp
    sftpro_idle_end = meas_peak - 10
    for j in range(meas_peak, md1_start, -1):
        if I[j] < idle_threshold:
            sftpro_idle_end = j
            break
    # Measurement starts at idle after last MD1 cycle (not at SFTPRO)
    md1_end = md1_start
    for j in range(sftpro_idle_end, md1_start, -1):
        if I[j] > idle_threshold:
            md1_end = j
            break
    return {"std": slice(0, md1_start), "md1": slice(md1_start, md1_end + 1), "meas": slice(md1_end + 1, len(I))}


def plot_colored_line(ax, x, y, c, cmap="coolwarm", linewidth=1.0, alpha=0.8):
    """Plot a line colored by c (e.g. turn index = time)."""
    if len(x) < 2:
        return None
    points = np.column_stack([x, y]).reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    norm = Normalize(vmin=c.min(), vmax=c.max())
    lc = LineCollection(segments, cmap=cmap, norm=norm, linewidths=linewidth, alpha=alpha)
    lc.set_array(c[:-1])
    ax.add_collection(lc)
    ax.autoscale()
    return lc


# Phase colors
PHASE_COLORS = {"std": "silver", "md1": "tab:purple", "meas": "tab:red"}
PHASE_LABELS = {"std": "Standardization", "md1": "MD1 cycles", "meas": "Measurement"}

# Load all data
turns = {"body": {}, "fringe": {}}
phases = {}
for name, cfg in SESSIONS.items():
    turns["body"][name] = load_turns("results_body", cfg)
    turns["fringe"][name] = load_turns("results_fringe", cfg)
    phases[name] = find_phases(turns["body"][name])
    n_std = phases[name]["std"].stop - phases[name]["std"].start
    n_md1 = phases[name]["md1"].stop - phases[name]["md1"].start
    n_meas = phases[name]["meas"].stop - phases[name]["meas"].start
    print(f"{name}: std={n_std}, md1={n_md1}, meas={n_meas} turns")

SEG_LABEL = {"body": "Body Segment", "fringe": "Fringe Segment"}

In [ ]:
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D


def lighten(color, amount=0.3):
    """Blend color toward white."""
    r, g, b = to_rgb(color)
    return (r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount)


def darken(color, amount=0.3):
    """Blend color toward black."""
    r, g, b = to_rgb(color)
    return (r * (1 - amount), g * (1 - amount), b * (1 - amount))


def plot_colored_line(ax, x, y, c, cmap="coolwarm", linewidth=1.0, alpha=0.8,
                      vmin=None, vmax=None):
    """Plot a line colored by c (e.g. turn index). Supports shared vmin/vmax."""
    if len(x) < 2:
        return None
    points = np.column_stack([x, y]).reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    norm = Normalize(vmin=c.min() if vmin is None else vmin,
                     vmax=c.max() if vmax is None else vmax)
    lc = LineCollection(segments, cmap=cmap, norm=norm, linewidths=linewidth, alpha=alpha)
    lc.set_array(c[:-1])
    ax.add_collection(lc)
    ax.autoscale()
    return lc


def assign_cycles(I_A, threshold=50):
    """Assign cycle index to each turn. New cycle at each upward crossing."""
    idx = np.zeros(len(I_A), dtype=int)
    cycle = 0
    for j in range(1, len(I_A)):
        if I_A[j] > threshold and I_A[j - 1] <= threshold:
            cycle += 1
        idx[j] = cycle
    return idx


def plot_cycle_segments(ax, x, y, rr, cycle_idx, lw=0.8, alpha=0.7):
    """Plot per-cycle with distinct color, light=ascending, dark=descending."""
    cmap = plt.cm.tab20
    n_cycles = cycle_idx.max() + 1
    for ci in range(n_cycles):
        mask = cycle_idx == ci
        if mask.sum() < 2:
            continue
        base = cmap(ci % 20)
        xs, ys, rs = x[mask], y[mask], rr[mask]
        direction = np.where(rs >= 0, 1, -1)
        changes = np.where(np.diff(direction) != 0)[0] + 1
        bounds = np.concatenate([[0], changes, [len(xs)]])
        asc_c, desc_c = lighten(base), darken(base)
        for j in range(len(bounds) - 1):
            s, e = bounds[j], bounds[j + 1]
            if e - s < 2:
                continue
            c = asc_c if direction[s] >= 0 else desc_c
            ax.plot(xs[s:e], ys[s:e], '-', linewidth=lw, color=c, alpha=alpha)


DIR_LEGEND = [
    Line2D([0], [0], color='silver', lw=2.5, label='ascending (light)'),
    Line2D([0], [0], color='dimgrey', lw=2.5, label='descending (dark)'),
]

# Transfer Function — MBB 2026-03-10 (Max-Speed + Idle)

Per-session plots (A-B-B-A protocol), body and fringe segments.
Three quantities: **B1(t)**, **B1(I)**, **TF(I)** — each as a 2×3 grid (rows = segments).

Two views per session:
1. **Temporal** — coolwarm colormap (blue=early → red=late), colorbar = turn index
2. **Per-cycle** — each current cycle a different color, lighter shade = ascending, darker = descending

In [ ]:
# === Version 1: Temporal evolution (coolwarm colormap) ===
for name in SESSION_NAMES:
    fig, axes = plt.subplots(3, 2, figsize=(12, 14))
    lc_ref = None

    for col, seg in enumerate(['body', 'fringe']):
        df = turns[seg][name]
        t, I, B = df['t_s'].values, df['I_A'].values, df['B1_T'].values
        c = df['turn'].values.astype(float)
        v0, v1 = 0, len(df) - 1

        # Row 0: B1 vs Time
        ax = axes[0, col]
        lc = plot_colored_line(ax, t, B, c, vmin=v0, vmax=v1)
        if lc and lc_ref is None:
            lc_ref = lc
        ax.set_ylabel('B1 (T)')
        ax.set_xlabel('Time (s)')
        ax.set_title(SEG_LABEL[seg])

        # Row 1: B1 vs I
        ax = axes[1, col]
        m = I > 10
        if m.sum() > 2:
            plot_colored_line(ax, I[m], B[m], c[m], vmin=v0, vmax=v1)
        ax.set_ylabel('B1 (T)')
        ax.set_xlabel('I (A)')

        # Row 2: TF vs I
        ax = axes[2, col]
        m2 = I > 100
        if m2.sum() > 2:
            tf = B[m2] / I[m2] * 1000
            plot_colored_line(ax, I[m2], tf, c[m2], vmin=v0, vmax=v1)
        ax.set_ylabel('TF = B1/I (T/kA)')
        ax.set_xlabel('I (A)')

    # Phase boundary: measurement starts at idle after last MD1 cycle
    ph = phases[name]
    for col, seg in enumerate(['body', 'fringe']):
        ax = axes[0, col]
        df_seg = turns[seg][name]
        if 0 < ph['meas'].start < len(df_seg):
            t_meas = df_seg['t_s'].values[ph['meas'].start]
            ax.axvline(t_meas, color='k', ls='--', lw=0.8, alpha=0.5)
            ax.text(t_meas, 0.98, ' Meas', fontsize=7, va='top',
                    transform=ax.get_xaxis_transform())

    if lc_ref:
        cb = fig.colorbar(lc_ref, ax=axes.ravel().tolist(), shrink=0.5, pad=0.02)
        cb.set_label('Turn index (time \u2192)')

    fig.suptitle(f'{name} \u2014 Temporal (blue=early, red=late)', fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
# === Version 2: Per-cycle coloring (each cycle distinct, light=asc, dark=desc) ===
for name in SESSION_NAMES:
    fig, axes = plt.subplots(3, 2, figsize=(12, 14))

    for col, seg in enumerate(['body', 'fringe']):
        df = turns[seg][name]
        t, I, B = df['t_s'].values, df['I_A'].values, df['B1_T'].values
        rr = df['rr'].values
        ci = assign_cycles(I)

        # Row 0: B1 vs Time
        ax = axes[0, col]
        plot_cycle_segments(ax, t, B, rr, ci)
        ax.set_ylabel('B1 (T)')
        ax.set_xlabel('Time (s)')
        ax.set_title(SEG_LABEL[seg])

        # Row 1: B1 vs I
        ax = axes[1, col]
        m = I > 10
        if m.sum() > 2:
            plot_cycle_segments(ax, I[m], B[m], rr[m], ci[m])
        ax.set_ylabel('B1 (T)')
        ax.set_xlabel('I (A)')

        # Row 2: TF vs I
        ax = axes[2, col]
        m2 = I > 100
        if m2.sum() > 2:
            tf = B[m2] / I[m2] * 1000
            plot_cycle_segments(ax, I[m2], tf, rr[m2], ci[m2])
        ax.set_ylabel('TF = B1/I (T/kA)')
        ax.set_xlabel('I (A)')

    # Phase boundary: measurement starts at idle after last MD1 cycle
    ph = phases[name]
    for col, seg in enumerate(['body', 'fringe']):
        ax = axes[0, col]
        df_seg = turns[seg][name]
        if 0 < ph['meas'].start < len(df_seg):
            t_meas = df_seg['t_s'].values[ph['meas'].start]
            ax.axvline(t_meas, color='k', ls='--', lw=0.8, alpha=0.5)
            ax.text(t_meas, 0.98, ' Meas', fontsize=7, va='top',
                    transform=ax.get_xaxis_transform())

    # Legend on each subplot
    for ax in axes.ravel():
        ax.legend(handles=DIR_LEGEND, fontsize=7, loc='upper right')

    fig.suptitle(f'{name} \u2014 Per-Cycle (each color = one cycle, light=asc, dark=desc)',
                 fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

## Animated Build-Up

Each session is drawn turn-by-turn (coolwarm coloring). Adjust `ANIM_STEP` and `ANIM_SESSIONS` to control speed and which sessions to animate.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import threading, time

# ── Settings ──────────────────────────────────────────────────────────
ANIM_TURNS_PER_FRAME = 3    # turns revealed per frame (lower = smoother)
ANIM_INTERVAL_MS = 25       # ms between frames (higher = slower)
ANIM_SESSIONS = SESSION_NAMES[:1]  # change to SESSION_NAMES for all

for name in ANIM_SESSIONS:
    # --- Pre-compute data per segment ---
    seg_data = {}
    global_vmax = 0
    for seg in ['body', 'fringe']:
        df = turns[seg][name]
        t = df['t_s'].values
        I = df['I_A'].values
        B = df['B1_T'].values
        c = df['turn'].values.astype(float)
        global_vmax = max(global_vmax, len(df) - 1)

        m1 = I > 10
        m2 = I > 100
        tf = np.full_like(B, np.nan)
        tf[m2] = B[m2] / I[m2] * 1000

        rows = {}
        for row_key, xv, yv, mask in [
            ('B1t', t, B, np.ones(len(t), dtype=bool)),
            ('B1I', I, B, m1),
            ('TFI', I, tf, m2),
        ]:
            xm, ym, cm = xv[mask], yv[mask], c[mask]
            pts = np.column_stack([xm, ym]).reshape(-1, 1, 2)
            segs = np.concatenate([pts[:-1], pts[1:]], axis=1) if len(xm) > 1 else np.empty((0, 2, 2))
            colors = cm[:-1] if len(cm) > 1 else np.array([])
            rows[row_key] = {
                'segs': segs, 'colors': colors, 'turn_vals': cm,
                'xlim': (xm.min(), xm.max()) if len(xm) > 1 else (0, 1),
                'ylim': (ym.min(), ym.max()) if len(ym) > 1 else (0, 1),
            }
        seg_data[seg] = rows

    norm = Normalize(vmin=0, vmax=global_vmax)
    n_turns = int(global_vmax) + 1
    total_frames = n_turns // ANIM_TURNS_PER_FRAME + 1

    # --- Create figure ---
    fig, axes = plt.subplots(3, 2, figsize=(12, 14))
    row_labels = ['B1t', 'B1I', 'TFI']
    ylabels = ['B1 (T)', 'B1 (T)', 'TF = B1/I (T/kA)']
    xlabels = ['Time (s)', 'I (A)', 'I (A)']
    title_obj = fig.suptitle(f'{name} \u2014 turn 0 / {n_turns}', fontsize=14)

    lc_dict = {}
    for ri, rk in enumerate(row_labels):
        for ci, seg in enumerate(['body', 'fringe']):
            ax = axes[ri, ci]
            rd = seg_data[seg][rk]
            dx = (rd['xlim'][1] - rd['xlim'][0]) * 0.05 or 1
            dy = (rd['ylim'][1] - rd['ylim'][0]) * 0.05 or 0.01
            ax.set_xlim(rd['xlim'][0] - dx, rd['xlim'][1] + dx)
            ax.set_ylim(rd['ylim'][0] - dy, rd['ylim'][1] + dy)
            ax.set_ylabel(ylabels[ri])
            ax.set_xlabel(xlabels[ri])
            if ri == 0:
                ax.set_title(SEG_LABEL[seg])
            lc = LineCollection([], cmap='coolwarm', norm=norm, linewidths=1.0, alpha=0.8)
            ax.add_collection(lc)
            lc_dict[(ri, ci)] = lc

    # Phase boundary on B1(t) row
    ph = phases[name]
    for ci, seg in enumerate(['body', 'fringe']):
        ax = axes[0, ci]
        df_seg = turns[seg][name]
        if 0 < ph['meas'].start < len(df_seg):
            t_meas = df_seg['t_s'].values[ph['meas'].start]
            ax.axvline(t_meas, color='k', ls='--', lw=0.8, alpha=0.5)
            ax.text(t_meas, 0.98, ' Meas', fontsize=7, va='top',
                    transform=ax.get_xaxis_transform())

    cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap='coolwarm'),
                      ax=axes.ravel().tolist(), shrink=0.5, pad=0.02)
    cb.set_label('Turn index (time \u2192)')
    fig.subplots_adjust(top=0.95, hspace=0.3, wspace=0.3)

    # --- Pre-compute segment counts per frame ---
    frame_counts = {}
    for ri, rk in enumerate(row_labels):
        for ci, seg in enumerate(['body', 'fringe']):
            tv = seg_data[seg][rk]['turn_vals']
            counts = np.empty(total_frames, dtype=int)
            for fi in range(total_frames):
                turn_show = min(fi * ANIM_TURNS_PER_FRAME, n_turns - 1)
                counts[fi] = max(int(np.searchsorted(tv, turn_show + 0.5)) - 1, 0)
            frame_counts[(ri, ci)] = counts

    # --- Draw a specific frame ---
    def _make_draw(fig, title_obj, lc_dict, seg_data, frame_counts,
                   row_labels, n_turns, step):
        def draw_frame(frame):
            turn_show = min(frame * step, n_turns - 1)
            title_obj.set_text(f'{name} \u2014 turn {turn_show} / {n_turns}')
            for ri, rk in enumerate(row_labels):
                for ci, seg in enumerate(['body', 'fringe']):
                    rd = seg_data[seg][rk]
                    n_segs = frame_counts[(ri, ci)][frame]
                    lc = lc_dict[(ri, ci)]
                    lc.set_segments(rd['segs'][:n_segs])
                    lc.set_array(rd['colors'][:n_segs])
            fig.canvas.draw_idle()
        return draw_frame

    draw_frame = _make_draw(fig, title_obj, lc_dict, seg_data,
                            frame_counts, row_labels, n_turns,
                            ANIM_TURNS_PER_FRAME)

    # --- Controls ---
    slider = widgets.IntSlider(
        value=0, min=0, max=total_frames - 1, step=1,
        description='Frame:', layout=widgets.Layout(width='60%'),
        continuous_update=True,
    )
    speed_slider = widgets.FloatSlider(
        value=1.0, min=0.2, max=5.0, step=0.1,
        description='Speed:', layout=widgets.Layout(width='30%'),
        readout_format='.1f',
    )
    btn_play    = widgets.Button(description='\u25b6 Play',    layout=widgets.Layout(width='80px'))
    btn_pause   = widgets.Button(description='\u23f8 Pause',   layout=widgets.Layout(width='80px'))
    btn_restart = widgets.Button(description='\u23ee Restart', layout=widgets.Layout(width='90px'))

    _state = {'paused': True, 'frame': 0}

    def on_slider_change(change):
        if _state['paused']:
            _state['frame'] = change['new']
            draw_frame(change['new'])

    def on_play(_):
        if _state['paused']:
            _state['paused'] = False
            if _state['frame'] >= total_frames - 1:
                _state['frame'] = 0
                slider.value = 0
            def _loop():
                while not _state['paused'] and _state['frame'] < total_frames:
                    draw_frame(_state['frame'])
                    slider.value = _state['frame']
                    _state['frame'] += 1
                    time.sleep(ANIM_INTERVAL_MS / 1000.0 / speed_slider.value)
                _state['paused'] = True
            threading.Thread(target=_loop, daemon=True).start()

    def on_pause(_):
        _state['paused'] = True

    def on_restart(_):
        _state['paused'] = True
        _state['frame'] = 0
        slider.value = 0
        draw_frame(0)

    slider.observe(on_slider_change, names='value')
    btn_play.on_click(on_play)
    btn_pause.on_click(on_pause)
    btn_restart.on_click(on_restart)

    controls = widgets.HBox([btn_restart, btn_play, btn_pause, slider, speed_slider])
    display(controls)
    draw_frame(0)
    plt.show()
    print(f"Animation for {name}: {total_frames} frames | "
          f"Buttons: play/pause/restart | Slider: scrub | Speed: 0.2x\u20135x")